Convert to a text-based Cortex demo
Available Cortex models: llama3.1-8b, mistral-7b, gemma-7b, llama3.1-70b, llama3.1-405b

In [ ]:
import streamlit as st
import pandas as pd
import warnings
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)

from snowflake.snowpark.context import get_active_session
from snowflake.cortex import Complete
session = get_active_session()

Define Model

In [ ]:
from snowflake.ml.model import custom_model
import logging

class TextProcessingModel(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.setLevel(logging.DEBUG)
        self._session = None

    def _get_session(self):
        if self._session is None:
            try:
                from snowflake.snowpark.context import get_active_session
                self._session = get_active_session()
            except:
                try:
                    import _snowflake
                    import snowflake.snowpark as snowpark
                    self._session = snowpark.Session.builder.configs(
                        {"connection": _snowflake.get_connection()}
                    ).create()
                except:
                    import snowflake.connector
                    conn = snowflake.connector.connect()
                    from snowflake.snowpark import Session
                    self._session = Session.builder.configs({"connection": conn}).create()
        return self._session

    def run_llm(self, text, task, model):
        prompts = {
            'summarize': f"Summarize concisely:\n\n{text}",
            'translate_es': f"Translate to Spanish:\n\n{text}",
            'translate_fr': f"Translate to French:\n\n{text}",
            'sentiment': f"Analyze sentiment (positive/negative/neutral):\n\n{text}",
            'qa': text
        }
        prompt = prompts.get(task, text)
        from snowflake.cortex import Complete
        return Complete(model, prompt, session=self._get_session())

    @custom_model.inference_api
    def transform(self, input_df: pd.DataFrame) -> pd.DataFrame:
        results = input_df.apply(
            lambda x: self.run_llm(x['TEXT_INPUT'], x['TASK'], x['MODEL']), axis=1
        )
        return pd.DataFrame({'RESULT': results})

mc = custom_model.ModelContext()
text_model = TextProcessingModel(context=mc)

Test Model

In [ ]:
input_df = pd.DataFrame([
    ["Snowflake Cortex Functions are being used in this notebook here.", "summarize", "llama3.1-8b"],
    ["Hope these work out well for our usecase.", "translate_es", "llama3.1-8b"],
    ["This feature exceeded my expectations!", "sentiment", "mistral-7b"],
], columns=['TEXT_INPUT', 'TASK', 'MODEL'])

output_df = text_model.transform(input_df)

for ix, row in pd.concat([input_df, output_df], axis=1).iterrows():
    with st.chat_message('ai'):
        st.markdown(f"**Task:** {row['TASK']} | **Model:** {row['MODEL']}")
        st.write(f"Input: {row['TEXT_INPUT']}")
        st.success(f"Output: {row['RESULT']}")

Register Model

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

reg = Registry(session=session, database_name="ER_DEMO", schema_name="PUBLIC")
model_signature = infer_signature(input_data=input_df, output_data=output_df)
print(model_signature)

In [ ]:
model_ref = reg.log_model(
    model_name="TEXT_PROCESSING",
    version_name="CORTEX_V3",
    model=text_model,
    signatures={"transform": model_signature},
    comment="Cortex LLM text processing [summarize, translate, sentiment]"
)

Inference Service - Container

WH INFERENCE

model_ref.run(
    input_df,
    function_name="transform"
)

SELECT ER_DEMO.PUBLIC.TEXT_PROCESSING!transform(
    object_construct(
        'TEXT_INPUT', 'Hello world',
        'TASK', 'translate_es', 
        'MODEL', 'llama3.1-8b'
    )
) AS RESULT;

In [ ]:
CREATE COMPUTE POOL IF NOT EXISTS TEXT_PROCESSING_POOL
    MIN_NODES = 1
    MAX_NODES = 1
    INSTANCE_FAMILY = CPU_X64_XS;

In [ ]:
inference_service = model_ref.create_service(
    service_name="ER_DEMO.PUBLIC.TEXT_PROCESSING",
    service_compute_pool="TEXT_PROCESSING_POOL",
    ingress_enabled=True
)

In [ ]:
model_ref.list_services()

Test Inference

In [ ]:
from snowflake.cortex import Complete

test_data = [
    ("Artificial intelligence is transforming industries worldwide. Westpac is also following suite.", "summarize", "llama3.1-8b"),
    ("Thank you for your help!", "translate_fr", "mistral-7b"),
]

def run_cortex(text, task, model):
    prompts = {
        'summarize': f"Summarize concisely:\n\n{text}",
        'translate_fr': f"Translate to French:\n\n{text}",
        'translate_es': f"Translate to Spanish:\n\n{text}",
        'sentiment': f"Analyze sentiment:\n\n{text}",
    }
    return Complete(model, prompts.get(task, text))

for text, task, model in test_data:
    result = run_cortex(text, task, model)
    with st.chat_message('ai'):
        st.markdown(f"**Task:** {task} | **Model:** {model}")
        st.write(f"Input: {text}")
        st.success(f"Output: {result}")

SQL Inference

In [ ]:
SELECT 
    'This demo worked well. Really happy how it turned out.' AS INPUT_TEXT,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-8b', 
        'Translate to Spanish:\n\nThis demo worked well. Really happy how it turned out.'
    ) AS RESULT;

Logging

In [ ]:
logs = session.call('SYSTEM$GET_SERVICE_LOGS', 'ER_DEMO.PUBLIC.TEXT_PROCESSING', '0', 'model-inference')
for line in logs.split('\n'):
    print(line)